In [0]:
from pathlib import Path

root = Path("/Workspace/Users/vijesh.rajan@hotmail.com/Meta_DataReplication")

print("Files and folders:")
for item in root.iterdir():
    print(item.name)

print("\nYAML files:")
for pattern in ("*.yml", "*.yaml"):
    for file in root.rglob(pattern):
        print(file)

In [0]:
from pathlib import Path
import yaml

config_path = Path(
    "/Workspace/Users/vijesh.rajan@hotmail.com/"
    "Meta_DataReplication/metadata/datasets.yml"
)

with config_path.open() as file:
    datasets = yaml.safe_load(file)["datasets"]

for name, config in datasets.items():
    print(f"{name} → {config['silver_table']}")

In [0]:
for name, config in datasets.items():
    df = spark.table(config["bronze_table"])

    missing = set(config["columns"]) - set(df.columns)

    if missing:
        raise ValueError(f"{name}: missing columns {sorted(missing)}")

    print(f"{name}: columns validated, {df.count()} records")

In [0]:
from pyspark.sql import functions as F

for name, config in datasets.items():
    df = spark.table(config["bronze_table"]).select(*config["columns"])
    keys = config["primary_keys"]

    for key in keys:
        if df.filter(F.col(key).isNull()).limit(1).count():
            raise ValueError(f"{name}: null primary key")

    if df.groupBy(*keys).count().filter("count > 1").limit(1).count():
        raise ValueError(f"{name}: duplicate primary keys")

    df = df.withColumn("updated_at", F.col("updated_at").cast("timestamp"))

    if df.filter(F.col("updated_at").isNull()).limit(1).count():
        raise ValueError(f"{name}: invalid updated_at")

    df.write.format("delta").mode("append").saveAsTable(
        config["silver_table"]
    )

    print(f"Created {config['silver_table']}")

In [0]:
%sql
SELECT 'employee' AS dataset, COUNT(*) AS records
FROM replication_catalog.silver.employee

UNION ALL

SELECT 'policy' AS dataset, COUNT(*) AS records
FROM replication_catalog.silver.policy;

In [0]:
manifest_path = Path('/Workspace/Users/vijesh.rajan@hotmail.com/Meta_DataReplication/metadata/batches/employee_002.yml')
base_path = "abfss://input@replstorageacc.dfs.core.windows.net/"

with manifest_path.open() as file:
    batch = yaml.safe_load(file)

assert batch["complete"], "Batch is incomplete"
assert batch["load_mode"] == "incremental", "Expected incremental batch"

config = datasets[batch["dataset"]]

paths = [
    f"{base_path.rstrip('/')}/{config['source_folder']}/{filename}"
    for filename in batch["files"]
]

incremental_df = (
    spark.read
    .option("header", True)
    .csv(paths)
)

display(incremental_df)

In [0]:

files = dbutils.fs.ls(
    "abfss://input@replstorageacc.dfs.core.windows.net/employee/"
)

for file in files:
    print(file.path)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

target_name = config["silver_table"]
target_df = spark.table(target_name)
keys = config["primary_keys"]
sequence = config["sequence_column"]

# Match the existing target's column types.
updates = incremental_df.select(*[
    F.col(field.name).cast(field.dataType).alias(field.name)
    for field in target_df.schema.fields
])

for column in keys + [sequence]:
    if updates.filter(F.col(column).isNull()).limit(1).count():
        raise ValueError(f"Missing or invalid value: {column}")

if updates.groupBy(*keys).count().filter("count > 1").limit(1).count():
    raise ValueError("Duplicate keys in this batch")

condition = " AND ".join(
    f"target.`{key}` = source.`{key}`" for key in keys
)

(
    DeltaTable.forName(spark, target_name)
    .alias("target")
    .merge(updates.alias("source"), condition)
    .whenMatchedUpdateAll(
        condition=f"source.`{sequence}` > target.`{sequence}`"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

display(spark.table(target_name).orderBy(*keys))

In [0]:
%sql
SELECT EmployeeID, COUNT(*) AS record_count
FROM replication_catalog.silver.employee
GROUP BY EmployeeID
ORDER BY EmployeeID;